### **`Data cleaning`**

In [28]:
# import dependencies
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit


In [29]:
# load data
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (15, 6)
plt.rcParams['font.size'] = 12

df = pd.read_csv('D:/PythonCode/DSEB65A_MachineLearningProject_Group5/data/Hanoi Daily 10 years.csv')

### **`Handling missing values for time-series problem`**

In [30]:
# handling Missing Values

def check_missing_values(df):
    """Return missing percentage for each column."""
    missing = df.isnull().mean() * 100
    return missing[missing > 0].sort_values(ascending=False)

missing_ratio = check_missing_values(df)
print("\nMissing Value Ratio (%):")
print(missing_ratio)

# focus on specific columns
if 'severerisk' in df.columns:
    print(f"\nsevererisk missing: {missing_ratio['severerisk']:.2f}%")
if 'preciptype' in df.columns:
    print(f"preciptype missing: {missing_ratio['preciptype']:.2f}%")


Missing Value Ratio (%):
severerisk    72.677596
preciptype    26.967213
dtype: float64

severerisk missing: 72.68%
preciptype missing: 26.97%


### **`Dropping redundant and zero-variance cols`**

In [31]:
# Constant and Zero-Variance Columns

def find_constant_columns(df):
    """Find columns with only one unique (non-null) value."""
    # This also finds numeric columns with zero variance
    return [col for col in df.columns if df[col].nunique(dropna=True) <= 1]

# find all constant columns
constant_cols = find_constant_columns(df)
print("\nConstant or Zero-Variance Columns Detected:")
print(constant_cols)

# (domain knowledge) other potential cols that should be dropped for future refactor
other_cols_to_drop = ['address', 'resolvedAddress', 'source', 'latitude', 'longitude', 'severerisk', 'stations']

# combine the lists and remove duplicates -> use set
all_cols_to_drop = list(set(constant_cols + other_cols_to_drop))

print("\nFull list of columns to be dropped:")
print(all_cols_to_drop)

# 4. drop the columns that actually exist in the dataframe
# PLEASE note that from now on we only process with cleaned_df for better later debugging
cleaned_df = df.drop(columns=[c for c in all_cols_to_drop if c in df.columns], errors='ignore')


Constant or Zero-Variance Columns Detected:
['name', 'preciptype', 'snow', 'snowdepth']

Full list of columns to be dropped:
['name', 'snow', 'source', 'stations', 'latitude', 'resolvedAddress', 'longitude', 'preciptype', 'severerisk', 'snowdepth', 'address']


In [32]:
# call check_missing_values() again for validation
remaining_missing = check_missing_values(cleaned_df)
if(remaining_missing.empty):
    print("\nNo more missing values")
else:
    print("\nRemaining Missing Value Ratio (%):")
    print(remaining_missing)


No more missing values


### **`Correcting Data Types`**

##### **Converting to timestamp datatype for time-series analysis**

In [33]:
# convert to timestamp data type
cleaned_df['datetime'] = pd.to_datetime(cleaned_df['datetime'])
cleaned_df['sunrise'] = pd.to_datetime(cleaned_df['sunrise'])
cleaned_df['sunset'] = pd.to_datetime(cleaned_df['sunset'])

##### **Multi-label Columns processing**  

**Cols**: `conditions`, `description`.

##### **Problem**:
|   | conditions           |
|---|----------------------|
| 0 | Rain, Partially cloudy |
| 1 | Rain, Overcast        |
| 2 | Rain, Overcast        |
| 3 | Rain, Partially cloudy |
| 4 | Rain, Partially cloudy |

```python
df['description'][0], df['conditions'][0]
>> ('Partly cloudy throughout the day with late afternoon rain.', 'Rain, Partially cloudy')
```

- `df['conditions']` does not contain a single category, but rather a string with multiple values separated by commas.

- For example: `'Rain, Partially cloudy'` means the day had both `Rain` **AND** `Partially cloudy` weather conditions.

- Additionally, the `description` column provides a textual summary of conditions column. For example, the description `'Partly cloudy throughout the day with late afternoon rain.'` corresponds to the conditions `'Rain, Partially cloudy'`.

##### **Solution:**

1. Use **Multi-Label Binarization**. This technique converts each individual label within a multi-label string into a separate binary column, enabling the model to interpret each label independently rather than as a combined entity. In pandas, the simplest and most effective approach is to use the `str.get_dummies()` method.
2. Drop `df['description']`.


In [34]:
df['description'][0], df['conditions'][0]

('Partly cloudy throughout the day with late afternoon rain.',
 'Rain, Partially cloudy')

In [35]:
# multi-label cols processing
# split 'condition' col into binary cols (sep=',')
conditions_dummies = cleaned_df['conditions'].str.get_dummies(sep=', ')

# concat to the processing dataframe
cleaned_df = pd.concat([cleaned_df, conditions_dummies], axis=1)

# now the original
cleaned_df.drop('conditions', axis=1, inplace=True)

# we can and must also drop df['description'],
# because it is just the more complicated version of df['condition'] which we have already handled
cleaned_df.drop('description', axis=1, inplace=True)

##### **One-hot Encoding**

The `icon` column is of object type and needs to be transformed into a numeric format for machine learning models. To do this, we apply **One-Hot Encoding**, a technique that converts categorical data into binary vectors.

This transformation ensures the model can process the data numerically and treat each icon category as a separate feature.


In [36]:
# apply one-hot encoding
icon_dummies = pd.get_dummies(cleaned_df['icon'], prefix='icon')

# concat
cleaned_df = pd.concat([cleaned_df, icon_dummies], axis=1)

# drop orignal one
cleaned_df.drop('icon', axis=1, inplace=True)

#### **`Main: Feature Engineering`**

##### **General Philosophy**
Our goal is to create new columns (features) that answer questions the model might "ask" in order to predict the temperature. For example:

*"Was it hot or cold yesterday?"* -> **Lag Features**

*"Did the temperature trend upward last week?"* -> **Rolling Features**

*"What season is it now?"* -> **Time-based Features**

*"Is the perceived temperature much different from the actual temperature?"* -> **Interaction Features**


In [37]:
# Các thành phần thời gian cơ bản
cleaned_df['month'] = cleaned_df['datetime'].dt.month
cleaned_df['day_of_year'] = cleaned_df['datetime'].dt.dayofyear
cleaned_df['day_of_week'] = cleaned_df['datetime'].dt.dayofweek
cleaned_df['week_of_year'] = cleaned_df['datetime'].dt.isocalendar().week.astype(int)
cleaned_df['year'] = cleaned_df['datetime'].dt.year

# daylight duration
cleaned_df['daylight_duration_sec'] = (cleaned_df['sunset'] - cleaned_df['sunrise']).dt.total_seconds()

# Feature mùa và mã hóa one-hot
season_map = {1: 'Winter', 2: 'Winter', 3: 'Spring', 4: 'Spring', 5: 'Spring',
              6: 'Summer', 7: 'Summer', 8: 'Summer', 9: 'Fall', 10: 'Fall',
              11: 'Fall', 12: 'Winter'}
cleaned_df['season'] = cleaned_df['month'].map(season_map)
season_dummies = pd.get_dummies(cleaned_df['season'], prefix='season')
cleaned_df = pd.concat([cleaned_df, season_dummies], axis=1)

# Mã hóa cyclical cho các biến thời gian
cleaned_df['month_sin'] = np.sin(2 * np.pi * cleaned_df['month'] / 12)
cleaned_df['month_cos'] = np.cos(2 * np.pi * cleaned_df['month'] / 12)
cleaned_df['day_of_year_sin'] = np.sin(2 * np.pi * cleaned_df['day_of_year'] / 365.25)
cleaned_df['day_of_year_cos'] = np.cos(2 * np.pi * cleaned_df['day_of_year'] / 365.25)


In [38]:
# Mã hóa hướng gió (0-360 độ)
cleaned_df['winddir_sin'] = np.sin(np.deg2rad(cleaned_df['winddir']))
cleaned_df['winddir_cos'] = np.cos(np.deg2rad(cleaned_df['winddir']))

# Mã hóa pha mặt trăng (0-1)
cleaned_df['moonphase_sin'] = np.sin(2 * np.pi * cleaned_df['moonphase'])
cleaned_df['moonphase_cos'] = np.cos(2 * np.pi * cleaned_df['moonphase'])


In [39]:
# lag features
lag_cols = ['temp', 'humidity', 'windspeed', 'cloudcover', 'precip']
for col in lag_cols:
    # Tạo lag 1, 2, 7 ngày
    for lag in [1, 2, 3,4,7, 14]:
        cleaned_df[f'{col}_lag_{lag}'] = cleaned_df[col].shift(lag)
    # Tạo feature chênh lệch so với ngày hôm qua
    cleaned_df[f'{col}_diff_1'] = cleaned_df[col].diff(periods=1)

# rolling: calculate on the lagged cols
for window in [3,5, 7, 14]:
    cleaned_df[f'temp_roll_mean_{window}'] = cleaned_df['temp'].shift(1).rolling(window=window).mean()
    cleaned_df[f'temp_roll_std_{window}'] = cleaned_df['temp'].shift(1).rolling(window=window).std()
    cleaned_df[f'humidity_roll_mean_{window}'] = cleaned_df['humidity'].shift(1).rolling(window=window).mean()



In [40]:
# Biên độ nhiệt trong ngày
cleaned_df['temp_range'] = cleaned_df['tempmax'] - cleaned_df['tempmin']

# Độ bão hòa hơi nước
cleaned_df['dew_point_depression'] = cleaned_df['temp'] - cleaned_df['dew']

# Vector gió (sử dụng các cột sin/cos đã tạo)
cleaned_df['wind_vector_ns'] = cleaned_df['windspeed'] * cleaned_df['winddir_cos']
cleaned_df['wind_vector_ew'] = cleaned_df['windspeed'] * cleaned_df['winddir_sin']

# Bức xạ mặt trời hiệu dụng
cleaned_df['effective_radiation'] = cleaned_df['solarradiation'] * (1 - cleaned_df['cloudcover'] / 100)

# Interaction
cleaned_df['humidity_temp_interact'] = cleaned_df['humidity'] * cleaned_df['temp']


In [41]:
# Danh sách các cột gốc cần xóa vì thông tin đã được chuyển hóa
cols_to_drop = [
    'datetime', 'sunrise', 'sunset', # Đã khai thác
    'winddir', 'moonphase',         # Đã mã hóa
    'season',                       # Đã mã hóa one-hot
    'name', 'stations', 'description', 'preciptype', 'icon', 'conditions', # Cột không cần thiết
    'severerisk', # Có quá nhiều giá trị thiếu
    'feelslike',
    'feelslikemax',
    'feelslikemin',
]

# Chỉ xóa các cột thực sự tồn tại
existing_cols_to_drop = [col for col in cols_to_drop if col in cleaned_df.columns]
cleaned_df.drop(columns=existing_cols_to_drop, inplace=True)
print(f"   - Dropped {len(existing_cols_to_drop)} unnecessary columns.")

# Xóa tất cả các dòng có NaN (chủ yếu do lag và rolling)
initial_rows = len(cleaned_df)
cleaned_df.dropna(inplace=True)
final_rows = len(cleaned_df)
print(f"   - Dropped {initial_rows - final_rows} rows with NaN values.")


   - Dropped 9 unnecessary columns.
   - Dropped 14 rows with NaN values.


#### 76 features are...
-> Lets go do some correlation analysis

### **`Correlation analysis`**

In [42]:
# Select numeric columns
num_df = cleaned_df.select_dtypes(include=[np.number])

# Compute Pearson correlation
corr_matrix = num_df.corr()

# Visualization function
def plot_correlation_heatmap(correlation_matrix, figsize=(20, 15), annot=False, title="Correlation Matrix"):
    """
    Vẽ heatmap tương quan. Mặc định tắt annot để dễ nhìn hơn với ma trận lớn.
    """
    mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))
    plt.figure(figsize=figsize)
    sns.heatmap(correlation_matrix,
                mask=mask,          # Luôn áp dụng mask
                cmap='coolwarm',
                center=0,
                annot=annot,        # Tùy chọn bật/tắt giá trị
                fmt=".2f",
                linewidths=.5)
    plt.title(title, fontsize=16)
    plt.tight_layout()
    plt.show()

In [43]:
# Sort top correlations with temp
top_corr = corr_matrix['temp'].abs().sort_values(ascending=False).head(40)
print("\nTop 40 Features Correlated with 'temp':")
print(top_corr)


Top 40 Features Correlated with 'temp':
temp                      1.000000
tempmax                   0.977416
tempmin                   0.969427
temp_lag_1                0.945086
temp_roll_mean_3          0.905448
dew                       0.902942
temp_roll_mean_5          0.885043
temp_roll_mean_7          0.874657
temp_lag_2                0.874333
sealevelpressure          0.867848
temp_roll_mean_14         0.864984
humidity_temp_interact    0.856874
temp_lag_3                0.824946
day_of_year_cos           0.817616
temp_lag_4                0.795626
daylight_duration_sec     0.763251
temp_lag_7                0.759511
temp_lag_14               0.750338
month_cos                 0.716122
solarradiation            0.615830
solarenergy               0.615616
uvindex                   0.580681
temp_roll_std_14          0.497333
month_sin                 0.463340
temp_range                0.394088
winddir_cos               0.389689
wind_vector_ns            0.384688
visibility    

In [44]:
# Handling multicolinearity

def find_highly_correlated_features(corr_matrix, threshold=0.95):
    """
    Find feature pairs with correlation above the specified threshold.
    """
    # Get the upper triangle of the correlation matrix
    upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

    # Find feature pairs with correlation > threshold
    highly_correlated_pairs = [
        (column, upper_tri[column].abs().idxmax(), upper_tri[column].abs().max())
        for column in upper_tri.columns if upper_tri[column].abs().max() > threshold
    ]

    # Sort for easier viewing
    sorted_pairs = sorted(highly_correlated_pairs, key=lambda x: x[2], reverse=True)

    return sorted_pairs


def drop_correlated_features(df, correlated_pairs, target_col):
    cols_to_drop = set()

    # Calculate correlation of all features with the target column once
    corr_with_target = df.corr()[target_col].abs()

    # Iterate through the pairs found
    for feat1, feat2, _ in correlated_pairs:
        # Get the correlation values with the target for each feature in the pair
        corr1 = corr_with_target.get(feat1, 0)
        corr2 = corr_with_target.get(feat2, 0)

        # Decide which feature to drop
        if corr1 > corr2:
            # Keep feat1, drop feat2
            cols_to_drop.add(feat2)
        else:
            # Keep feat2, drop feat1
            cols_to_drop.add(feat1)

    # Drop the identified columns
    df_reduced = df.drop(columns=list(cols_to_drop))

    print(f"\nDropped {len(cols_to_drop)} features due to multicollinearity.")
    if cols_to_drop:
        print("- Dropped columns:", list(cols_to_drop))

    return df_reduced, cols_to_drop

In [45]:
correlated_pairs = find_highly_correlated_features(corr_matrix, threshold=0.70)

print("\n feature pairs with very high correlation (> 0.90):")
for feat1, feat2, corr_val in correlated_pairs:
    print(f"- {feat1:<25} and {feat2:<25} : {corr_val:.4f}")

df_final, dropped_cols = drop_correlated_features(df=cleaned_df,
                                                  correlated_pairs=correlated_pairs,
                                                  target_col='temp')


 feature pairs with very high correlation (> 0.90):
- Rain                      and precipprob                : 1.0000
- solarenergy               and solarradiation            : 0.9999
- day_of_year               and month                     : 0.9965
- dew_point_depression      and humidity                  : 0.9915
- temp_roll_mean_7          and temp_roll_mean_5          : 0.9896
- temp_roll_mean_3          and temp_lag_2                : 0.9896
- humidity_temp_interact    and dew                       : 0.9889
- day_of_year_cos           and daylight_duration_sec     : 0.9854
- month_cos                 and daylight_duration_sec     : 0.9839
- temp_roll_mean_5          and temp_roll_mean_3          : 0.9824
- temp                      and tempmax                   : 0.9774
- week_of_year              and day_of_year               : 0.9754
- temp_roll_mean_14         and temp_roll_mean_7          : 0.9699
- uvindex                   and solarradiation            : 0.9612
- wind_ve

### **Label Creating**

In [46]:
df_final, dropped_cols = drop_correlated_features(df=cleaned_df,
                                                  correlated_pairs=correlated_pairs,
                                                  target_col='temp')
#Tạo 5 objective feature tương ứng với 5 ngày cần dự đoán
horizons = [1, 2, 3, 4, 5]
for h in horizons:
    df_final[f'target_temp_t+{h}'] = df_final['temp'].shift(-h)
df_final = df_final.dropna()



Dropped 37 features due to multicollinearity.
- Dropped columns: ['temp_roll_mean_7', 'month_cos', 'temp_lag_2', 'temp_lag_1', 'temp_lag_3', 'month', 'day_of_year_sin', 'temp_lag_7', 'dew_point_depression', 'humidity_temp_interact', 'temp_lag_4', 'uvindex', 'temp_roll_mean_5', 'Overcast', 'cloudcover', 'wind_vector_ns', 'sealevelpressure', 'humidity_roll_mean_5', 'tempmin', 'humidity_roll_mean_7', 'humidity_lag_1', 'humidity_lag_2', 'week_of_year', 'temp_roll_mean_14', 'humidity_roll_mean_3', 'humidity', 'dew', 'humidity_lag_3', 'precipprob', 'solarenergy', 'daylight_duration_sec', 'winddir_sin', 'temp_lag_14', 'tempmax', 'temp_roll_std_5', 'humidity_lag_4', 'day_of_year']


In [47]:
print(df_final.columns)
print(f"Number of columns in df_final: {df_final.shape[1]}")

Index(['temp', 'precip', 'precipcover', 'windgust', 'windspeed', 'visibility',
       'solarradiation', 'Clear', 'Partially cloudy', 'Rain', 'icon_clear-day',
       'icon_cloudy', 'icon_partly-cloudy-day', 'icon_rain', 'day_of_week',
       'year', 'season_Fall', 'season_Spring', 'season_Summer',
       'season_Winter', 'month_sin', 'day_of_year_cos', 'winddir_cos',
       'moonphase_sin', 'moonphase_cos', 'temp_diff_1', 'humidity_lag_7',
       'humidity_lag_14', 'humidity_diff_1', 'windspeed_lag_1',
       'windspeed_lag_2', 'windspeed_lag_3', 'windspeed_lag_4',
       'windspeed_lag_7', 'windspeed_lag_14', 'windspeed_diff_1',
       'cloudcover_lag_1', 'cloudcover_lag_2', 'cloudcover_lag_3',
       'cloudcover_lag_4', 'cloudcover_lag_7', 'cloudcover_lag_14',
       'cloudcover_diff_1', 'precip_lag_1', 'precip_lag_2', 'precip_lag_3',
       'precip_lag_4', 'precip_lag_7', 'precip_lag_14', 'precip_diff_1',
       'temp_roll_mean_3', 'temp_roll_std_3', 'temp_roll_std_7',
       'temp_

### **Data Spliting**

In [48]:
from sklearn.model_selection import train_test_split

# Giả sử df_final là DataFrame của bạn trước khi split
X = df_final.drop([f'target_temp_t+{h}' for h in horizons], axis=1)
y = df_final[[f'target_temp_t+{h}' for h in horizons]]

# Split the data into train (70%) and temp (30%)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, shuffle=False
)
# Split the temp data into validation (15%) and test (15%)
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, shuffle=False
)

### **Feature Selecting**

In [50]:
from catboost import CatBoostRegressor
from sklearn.inspection import permutation_importance
import pandas as pd
import numpy as np
from sklearn.multioutput import MultiOutputRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Huấn luyện mô hình LightGBM
# Tạo pipeline với StandardScaler và LGBMRegressor
pipeline = Pipeline([
    ('scaler', RobustScaler()),
    ('model', MultiOutputRegressor(CatBoostRegressor(random_state=42, verbose=0)))
])
pipeline.fit(X_train, y_train)

# Tính toán Permutation Importance
print("\n--- Đánh giá tầm quan trọng của đặc trưng theo hoán vị (Permutation Importance) ---")
result = permutation_importance(
    pipeline, X_test, y_test,
    n_repeats=10,  # Số lần lặp lại việc xáo trộn cho mỗi đặc trưng
    random_state=42,
    n_jobs=-1,  # Sử dụng tất cả các lõi CPU
    scoring='neg_root_mean_squared_error'  # Đánh giá bằng RMSE (phủ định)
)

# Lấy tên đặc trưng
if hasattr(X_test, 'columns'):
    feature_names_pi = X_test.columns.tolist()
else:
    feature_names_pi = [f"feature_{i}" for i in range(X_test.shape[1])]

# Tạo DataFrame
permutation_importance_df = pd.DataFrame({
    'Feature': feature_names_pi,
    'Importance_Mean': result.importances_mean,
    'Importance_Std': result.importances_std
})

# Sắp xếp theo tầm quan trọng giảm dần
permutation_importance_df = permutation_importance_df.sort_values(by='Importance_Mean', ascending=False)

# In top 10 đặc trưng quan trọng nhất
print("\nTop 10 đặc trưng quan trọng nhất theo Permutation Importance:")
print(permutation_importance_df.head(10))



--- Đánh giá tầm quan trọng của đặc trưng theo hoán vị (Permutation Importance) ---

Top 10 đặc trưng quan trọng nhất theo Permutation Importance:
                Feature  Importance_Mean  Importance_Std
21      day_of_year_cos         1.434814        0.056815
0                  temp         0.740550        0.027109
50     temp_roll_mean_3         0.152688        0.021598
19        season_Winter         0.136228        0.014877
56       wind_vector_ew         0.018837        0.003889
57  effective_radiation         0.014912        0.004895
22          winddir_cos         0.013416        0.004776
4             windspeed         0.009945        0.002957
25          temp_diff_1         0.009526        0.001456
16          season_Fall         0.009246        0.001602


## Update train, valid, test with permuted feature

In [55]:
# Lọc các cột trong X_train và X_test dựa trên permutation_importance_df['Feature']
threshold = 0.001
selected_features = permutation_importance_df[permutation_importance_df['Importance_Mean'] > threshold]['Feature'].tolist()
# Tạo X_train2 và X_test2 chỉ với các cột được chọn
X_train_permuted = X_train[selected_features]
X_valid_permuted = X_valid[selected_features]
X_test_permuted = X_test[selected_features]

print("Shape of X_train permuted:", X_train_permuted.shape)
print("Shape of X valid permuted", X_valid_permuted.shape)
print("Shape of X_test permuted", X_test_permuted.shape)

Shape of X_train permuted: (2548, 26)
Shape of X valid permuted (546, 26)
Shape of X_test permuted (547, 26)


## Optuna hypeparm

In [57]:
import optuna
from lightgbm import LGBMRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

model_for_selection = MultiOutputRegressor(CatBoostRegressor(random_state=42, verbose=0))

def objective(trial):
    """
    Optuna objective: tối ưu siêu tham số cho LGBMRegressor (MultiOutput).
    """
    param = {
            "iterations": trial.suggest_int("iterations", 100, 3000),
            "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
            "depth": trial.suggest_int("depth", 4, 10),
            "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-3, 10.0, log=True),
            "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
            "random_strength": trial.suggest_float("random_strength", 1e-3, 10.0, log=True),
            "border_count": trial.suggest_int("border_count", 32, 255),
            "grow_policy": trial.suggest_categorical("grow_policy", ["SymmetricTree", "Depthwise", "Lossguide"]),
            "random_seed": 42,
            "verbose": 0
        }

    model = CatBoostRegressor(**param)

    multi_model = MultiOutputRegressor(model)
    multi_model.fit(X_train_permuted, y_train)

    y_pred = multi_model.predict(X_valid_permuted)

    r2 = r2_score(y_valid, y_pred, multioutput='uniform_average')
    return r2

def print_best_trial(study, trial):
    """
    Callback để in thông tin về trial tốt nhất sau mỗi lần chạy.
    """
    print("\n------------------------------------------------")
    print(f"Trial hiện tại: {trial.number}")
    print(f"  Giá trị RMSE: {trial.value:.4f}")
    print("Trial tốt nhất hiện tại:")
    print(f"  Giá trị RMSE tốt nhất: {study.best_value:.4f}")
    print("  Các siêu tham số tốt nhất: ")
    for key, value in study.best_params.items():
        print(f"    {key}: {value}")
    print("------------------------------------------------")

# Tạo Optuna study.
study = optuna.create_study(direction="maximize",
                            pruner=optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=30))

print("Bắt đầu quá trình tối ưu hóa siêu tham số LGBMRegressor với Optuna...")
study.optimize(objective, n_trials=30, n_jobs=-1, callbacks=[print_best_trial])

# In kết quả tốt nhất tìm được.
print("\n------------------------------------------------")
print("Kết quả tối ưu hóa hoàn tất.")
print("Trial tốt nhất:")
print(f"  Giá trị R2 tốt nhất trên tập validation: {study.best_value:.4f}")
print("  Các siêu tham số tốt nhất: ")
for key, value in study.best_params.items():
    print(f"    {key}: {value}")
print("------------------------------------------------")


[I 2025-10-24 10:57:09,709] A new study created in memory with name: no-name-48aeb34a-0470-49f5-9e9f-b719ac3751b2


Bắt đầu quá trình tối ưu hóa siêu tham số LGBMRegressor với Optuna...


[I 2025-10-24 10:59:14,558] Trial 4 finished with value: 0.7745372893093503 and parameters: {'iterations': 668, 'learning_rate': 0.003714667774648688, 'depth': 6, 'l2_leaf_reg': 0.05726054936305721, 'bagging_temperature': 0.9248666837891144, 'random_strength': 0.3301282569511556, 'border_count': 213, 'grow_policy': 'Lossguide'}. Best is trial 4 with value: 0.7745372893093503.



------------------------------------------------
Trial hiện tại: 4
  Giá trị RMSE: 0.7745
Trial tốt nhất hiện tại:
  Giá trị RMSE tốt nhất: 0.7745
  Các siêu tham số tốt nhất: 
    iterations: 668
    learning_rate: 0.003714667774648688
    depth: 6
    l2_leaf_reg: 0.05726054936305721
    bagging_temperature: 0.9248666837891144
    random_strength: 0.3301282569511556
    border_count: 213
    grow_policy: Lossguide
------------------------------------------------


[I 2025-10-24 10:59:36,949] Trial 3 finished with value: 0.765949403407276 and parameters: {'iterations': 2712, 'learning_rate': 0.0098611803662678, 'depth': 4, 'l2_leaf_reg': 0.9985105680485756, 'bagging_temperature': 0.31748611389072945, 'random_strength': 0.03527180503577465, 'border_count': 135, 'grow_policy': 'Depthwise'}. Best is trial 4 with value: 0.7745372893093503.


...


------------------------------------------------
Trial hiện tại: 17
  Giá trị RMSE: 0.7634
Trial tốt nhất hiện tại:
  Giá trị RMSE tốt nhất: 0.7793
  Các siêu tham số tốt nhất: 
    iterations: 1493
    learning_rate: 0.0023104557446006536
    depth: 8
    l2_leaf_reg: 0.0010274933534587323
    bagging_temperature: 0.03729410092829827
    random_strength: 0.007336229178601035
    border_count: 34
    grow_policy: SymmetricTree
------------------------------------------------


[I 2025-10-24 11:23:32,139] Trial 13 finished with value: 0.7732043003824958 and parameters: {'iterations': 2794, 'learning_rate': 0.02138754349899455, 'depth': 10, 'l2_leaf_reg': 6.114091351914441, 'bagging_temperature': 0.6554937868880367, 'random_strength': 0.005588635009000316, 'border_count': 143, 'grow_policy': 'SymmetricTree'}. Best is trial 27 with value: 0.7793090922346521.



------------------------------------------------
Trial hiện tại: 13
  Giá trị RMSE: 0.7732
Trial tốt nhất hiện tại:
  Giá trị RMSE tốt nhất: 0.7793
  Các siêu tham số tốt nhất: 
    iterations: 1493
    learning_rate: 0.0023104557446006536
    depth: 8
    l2_leaf_reg: 0.0010274933534587323
    bagging_temperature: 0.03729410092829827
    random_strength: 0.007336229178601035
    border_count: 34
    grow_policy: SymmetricTree
------------------------------------------------


[I 2025-10-24 11:28:34,475] Trial 6 finished with value: 0.7672999699644889 and parameters: {'iterations': 2100, 'learning_rate': 0.05149868718421896, 'depth': 9, 'l2_leaf_reg': 0.012476428539936746, 'bagging_temperature': 0.15899662071853704, 'random_strength': 0.21117135492623973, 'border_count': 243, 'grow_policy': 'Depthwise'}. Best is trial 27 with value: 0.7793090922346521.



------------------------------------------------
Trial hiện tại: 6
  Giá trị RMSE: 0.7673
Trial tốt nhất hiện tại:
  Giá trị RMSE tốt nhất: 0.7793
  Các siêu tham số tốt nhất: 
    iterations: 1493
    learning_rate: 0.0023104557446006536
    depth: 8
    l2_leaf_reg: 0.0010274933534587323
    bagging_temperature: 0.03729410092829827
    random_strength: 0.007336229178601035
    border_count: 34
    grow_policy: SymmetricTree
------------------------------------------------

------------------------------------------------
Kết quả tối ưu hóa hoàn tất.
Trial tốt nhất:
  Giá trị R2 tốt nhất trên tập validation: 0.7793
  Các siêu tham số tốt nhất: 
    iterations: 1493
    learning_rate: 0.0023104557446006536
    depth: 8
    l2_leaf_reg: 0.0010274933534587323
    bagging_temperature: 0.03729410092829827
    random_strength: 0.007336229178601035
    border_count: 34
    grow_policy: SymmetricTree
------------------------------------------------


## Test trên tập test

In [59]:
best_params = study.best_params

# Huấn luyện lại model với các tham số tốt nhất
best_model = MultiOutputRegressor(CatBoostRegressor(**best_params, random_state=42))
best_model.fit(X_train_permuted, y_train)

# Dự đoán trên tập test
y_pred = best_model.predict(X_test_permuted)

# Hiển thị một vài giá trị dự đoán và giá trị thực tế để so sánh
comparison = pd.DataFrame({
    'Actual': y_test.values.flatten(),
    'Predicted': y_pred.flatten()
}).head(10)

print("So sánh giá trị thực tế và dự đoán:")
print(comparison)

# Tính toán các chỉ số đánh giá
r2 = r2_score(y_test, y_pred, multioutput='uniform_average')
rmse = np.sqrt(mean_squared_error(y_test, y_pred, multioutput='uniform_average'))
mae = mean_absolute_error(y_test, y_pred, multioutput='uniform_average')

print(f"\nĐánh giá trên tập test:")
print(f"R² trung bình: {r2:.4f}")
print(f"RMSE trung bình: {rmse:.4f}")
print(f"MAE trung bình: {mae:.4f}")

0:	learn: 5.1339838	total: 43.7ms	remaining: 1m 5s
1:	learn: 5.1235533	total: 50.3ms	remaining: 37.5s
2:	learn: 5.1131288	total: 55.8ms	remaining: 27.7s
3:	learn: 5.1027170	total: 61.3ms	remaining: 22.8s
4:	learn: 5.0923355	total: 66.5ms	remaining: 19.8s
5:	learn: 5.0819609	total: 72.8ms	remaining: 18s
6:	learn: 5.0716019	total: 77.7ms	remaining: 16.5s
7:	learn: 5.0612668	total: 82.4ms	remaining: 15.3s
8:	learn: 5.0508992	total: 87.9ms	remaining: 14.5s
9:	learn: 5.0405901	total: 92.7ms	remaining: 13.8s
10:	learn: 5.0303176	total: 97.7ms	remaining: 13.2s
11:	learn: 5.0201164	total: 102ms	remaining: 12.6s
12:	learn: 5.0099357	total: 109ms	remaining: 12.4s
13:	learn: 4.9997635	total: 118ms	remaining: 12.4s
14:	learn: 4.9896003	total: 125ms	remaining: 12.3s
15:	learn: 4.9794669	total: 132ms	remaining: 12.2s
16:	learn: 4.9693563	total: 138ms	remaining: 11.9s
17:	learn: 4.9592626	total: 142ms	remaining: 11.6s
18:	learn: 4.9491676	total: 147ms	remaining: 11.4s
19:	learn: 4.9391500	total: 153m